In [5]:
import pandas as pd
import dimod
from dwave.samplers import SimulatedAnnealingSampler

# Optimization problems

Optimization problems are a central topic in computer science, mathematics and quantum computing. In general, they consist of finding the best possible solution among many alternatives, according to a given objective function. This objective can represent a cost to minimize, a profit to maximize, or any quantity that measures how good a solution is.

In this notebook, we focus on binary optimization problems, where the decision variables can only take the values 0 or 1. These problems are especially relevant because many real situations, such as selecting items, assigning resources or finding optimal configurations, can be expressed in this form. To formulate them, we will use the QUBO framework and its relation with the Ising model.

# Quadratic Unconstrained Binary Optimization

The Quadratic Unconstrained Binary Optimization (QUBO) framework helps us formulate optimization problems using binary variables $x_i\in\{0,1\}$. The cost function to minimize is given by:

$$
f(x) = \sum_{i<j}Q_{ij}x_ix_j + \sum_i Q_{ii}x_i = x^TQx
$$

where $Q_{ij}$ is the strength between the variable $i$ and $j$.

The term "unconstrained" does not mean that this formulation cannot be used for problems with restrictions. The procedure is to include the constraints inside the objective function by adding penalty terms. These penalties increase the value of the function when a constraint is violated, so the minimization process naturally favors solutions that satisfy the original restrictions.

In general, an optimization problem can be translated into QUBO by following three steps. First, the decision variables are encoded as binary variables. Then, the objective function is written in terms of these variables. Finally, each constraint is converted into a quadratic penalty term and added to the objective function. Therefore, the final QUBO function contains both the original goal of the problem and the penalties that enforce valid solutions:

$$
f_{QUBO}(x) = f_{objective}(x) + \lambda f_{penalty}(x)
$$

where $\lambda$ is a positive parameter that controls the importance of satisfying the constraints. The value of $\lambda$ must be chosen carefully. If it is too small, the optimizer may prefer solutions that improve the original objective function but violate the constraints. On the other hand, if it is too large, the penalty terms can dominate the optimization and make the objective function less relevant. In practice, $\lambda$ is usually selected large enough to ensure that any constraint violation is more costly than the possible improvement obtained in the original objective function.

Let's first consider linear constraints of the form:

$$
Ax= b
$$

It's very easy to deal with this constraints since we can directly add them to the cost function as:

$$
f_{QUBO}(x) = f_{objective}(x) + \lambda \|Ax-b\|^2
$$

Now, let's consider inequality constraints of the form:

$$
Ax \leq b
$$

The method to translate inequality constraints into equality constraints is to add slack variables. These variables represent the difference between the right-hand side and the left-hand side of the inequality.

For example, let's consider the constraint:

$$
3x_0 - x_1 + 3x_2 \leq 5
$$

First, we need to find the minimum value that the left-hand side can take. To do this, we set the variables with positive coefficients to 0 and the variables with negative coefficients to 1. In this case, the minimum value is:

$$
3\cdot 0 - 1 + 3\cdot 0 = -1
$$

Therefore, the slack variable must be able to represent values from 0 to 6, since:

$$
5 - (-1) = 6
$$

Since 3 bits are enough to represent values up to 6, we introduce three binary slack variables $y_0$, $y_1$ and $y_2$:

$$
s = y_0 + 2y_1 + 4y_2
$$

Then, the inequality can be rewritten as the following equality:

$$
3x_0 - x_1 + 3x_2 + y_0 + 2y_1 + 4y_2 = 5
$$

Finally, we have an equality constraint, so we can add it to the QUBO objective as a penalty term:

$$
f_{QUBO}(x) = f_{objective}(x) + \lambda (3x_0 - x_1 + 3x_2 + y_0 + 2y_1 + 4y_2 - 5)^2
$$



# Ising Model


The Ising model is a mathematical model originally introduced to describe magnetic systems, especially the interaction between particles with spin. These spins are usually arranged in a lattice, and each spin can point in one of two possible directions. The energy of the system depends on the interaction between neighboring spins and on the effect of external local magnetic fields.

The Hamiltonian, which represents the energy of the system, is given by:

$$
H(s) = \sum_{i<j} J_{ij}s_is_j+\sum_i h_is_i
$$

where $s_i \in \{-1,1\}$ represents the spin variables, $J_{ij}$ is the strength of the interaction between spins $i$ and $j$, and $h_i$ is the local magnetic field acting on spin $i$.

This formulation is of great interest because many optimization problems can be expressed as particular cases of the Ising model. Moreover, since the spin variables take values $s_i \in \{-1,1\}$, the formulation can be naturally translated to a quantum computer, where the eigenvalues of the Pauli-$Z$ operator also take these values.

The Ising model is closely related to the QUBO formulation. The main difference is that QUBO uses binary variables $x_i \in \{0,1\}$, while the Ising model uses spin variables $s_i \in \{-1,1\}$. Both formulations can be transformed into each other using the change of variables:

$$
s_i = 2x_i - 1
$$

or equivalently,

$$
x_i = \frac{s_i+1}{2}
$$

Therefore, a QUBO problem can be rewritten as an Ising Hamiltonian by substituting this relation into the objective function and grouping the resulting terms. Both formulations represent the same optimization problem, up to an additive constant that does not affect the minimum.

# Quantum annealing

Quantum annealing is a quantum optimization technique used to find low-energy solutions of a problem Hamiltonian. The main idea is to encode the optimization problem into a Hamiltonian whose ground state represents the optimal solution. Then, the system is initialized in an easy-to-prepare ground state and slowly evolved towards the problem Hamiltonian. If the evolution is slow enough, the system tends to remain close to the ground state, making it possible to recover a good solution when the final state is measured.

The annealing process is usually described by a time-dependent Hamiltonian:

$$
H(t) = A(t)H_0 + B(t)H_1
$$

where $H_0$ is the initial Hamiltonian, whose ground state is easy to prepare, and $H_1$ is the final Hamiltonian, which encodes the optimization problem. The functions $A(t)$ and $B(t)$ control the evolution: at the beginning, $A(t)$ is large and $B(t)$ is close to zero, while at the end $A(t)$ goes to zero and $B(t)$ becomes dominant.

For the final Hamiltonian, we usually take:

$$
H_1 = -\sum_{j,k}J_{jk}Z_jZ_k -\sum_j h_j Z_j
$$

which is the quantum version of the Ising hamiltonian previously introduced.

For the initial Hamiltonian, we usually take:

$$
H_0 = -\sum_{j=0}^{n-1}X_j
$$

whose ground state is:

$$
|\psi_0\rangle = |+\rangle ^{\otimes n}
$$

That way, the time-dependent Hamiltonian of the annealing process is given by:

$$
H(t)=-A(t) \sum_{j=0}^{n-1}X_j - B(t)\sum_{j,k}J_{jk}Z_jZ_k - B(t)\sum_j h_j Z_j
$$

In order to solve optimization problems with quantum annealing we can use the Ocean library

# Max-Cut problem

The Max-Cut problem is a classical optimization problem defined on a graph. Given a graph with vertices and edges, the goal is to divide the vertices into two different groups in such a way that the number of edges connecting vertices from different groups is maximized. In other words, we want to find the partition of the graph that “cuts” as many edges as possible.

The Max-Cut problem can be naturally formulated using the Ising model. Given a graph, we assign one spin variable $s_i \in \{-1,1\}$ to each vertex. The value of the spin indicates which of the two groups the vertex belongs to. Therefore, two vertices are in different groups when their spins have opposite signs.

For an edge $(i,j)$, the term $s_i s_j$ is equal to $1$ if both vertices are in the same group, and equal to $-1$ if they are in different groups. Since Max-Cut tries to maximize the number of edges connecting different groups, we want to favor configurations where $s_i s_j = -1$. This can be done by minimizing the following Ising Hamiltonian:

$$
H(s) = \sum_{(i,j)\in E} s_i s_j
$$

where $E$ is the set of edges.

In [6]:
# Define the graph
nodes = ["A", "B", "C", "D", "E", "F"]

edges = [
    ("A", "B"),
    ("A", "C"),
    ("A", "E"),
    ("B", "D"),
    ("B", "E"),
    ("C", "D"),
    ("C", "E"),
    ("D", "E"),
    ("D", "F"),
    ("E", "F")
]

# Create an empty binary quadratic model for Max-Cut using Ising variables
bqm = dimod.BinaryQuadraticModel({}, {}, 0.0, dimod.SPIN)

# Add the Ising interaction terms
for u, v in edges:
    bqm.add_quadratic(u, v, 1)

In [7]:
# Use the Simulated Annealing Sampler to find a solution
sampler = SimulatedAnnealingSampler()
solution = sampler.sample(bqm, num_reads=1000)

# Aggregate the results to find the best solution
solution = solution.aggregate()
best = solution.first
best_energy = best.energy
df = solution.to_pandas_dataframe()
df_sorted = df.sort_values("energy", ascending=True)

# Print the best energy and the first 5 solutions
print(30 * "#")
print("Best energy found:", best_energy)
print(30 * "#")
print("First 5 solutions:")
print(df_sorted.head(5))

##############################
Best energy found: -6.0
##############################
First 5 solutions:
   A  B  C  D  E  F  energy  num_occurrences
0 -1  1  1 -1 -1  1    -6.0              488
1  1 -1 -1  1  1 -1    -6.0              508
2  1  1  1 -1 -1  1    -4.0                2
3 -1 -1 -1  1  1 -1    -4.0                2


The simulated annealing sampler found several configurations with the same minimum energy, $E=-6.0$. This means that there are multiple equivalent optimal partitions of the graph for the Max-Cut problem.

Since the Ising Hamiltonian gives lower energy when connected nodes have opposite spins, these solutions correspond to partitions where many edges are cut. Therefore, the sampler successfully found good Max-Cut solutions by minimizing the Ising energy.

# Travelling Salesman Problem

The Traveling Salesman Problem (TSP) is a classical optimization problem defined on a weighted graph. Given a set of cities and the distance between each pair of them, the goal is to find the shortest possible route that visits every city exactly once and returns to the starting point.

To formulate the TSP, we usually introduce binary variables $x_{i,t} \in \{0,1\}$, where $x_{i,t}=1$ means that city $i$ is visited at position $t$ in the route. The cost of the route can then be written as:

$$
H_{cost}(x) = \sum_{t=0}^{n-1}\sum_{i,j} d_{i,j}x_{i,t}x_{j,t+1}
$$

where $d_{i,j}$ is the distance between cities $i$ and $j$ and $t+1$ is modulo $N$.

However, we also need to enforce two constraints: each city must be visited exactly once, and each position in the route must contain exactly one city. These constraints can be added as penalty terms:

$$
H_{penalty}(x) = \sum_i \left(\sum_t x_{i,t}-1\right)^2 + \sum_t \left(\sum_i x_{i,t}-1\right)^2
$$

Therefore, the final Hamiltonian is:

$$
H(x) = H_{cost}(x) + \lambda H_{penalty}(x)
$$

This formulation is a QUBO problem, and it can be converted into an Ising model using the relation:

$$
x_i = \frac{s_i+1}{2}
$$  

The cost Hamiltonian becomes:

$$
H_{cost}(s) =
\sum_{t=0}^{n-1}\sum_{i,j} d_{i,j}
\left(\frac{s_{i,t}+1}{2}\right)
\left(\frac{s_{j,t+1}+1}{2}\right)
$$

or equivalently:

$$
H_{cost}(s) =
\frac{1}{4}
\sum_{t=0}^{n-1}\sum_{i,j} d_{i,j}
\left(1+s_{i,t}+s_{j,t+1}+s_{i,t}s_{j,t+1}\right)
$$

The constraints can be transformed in the same way:

$$
H_{penalty}(s) =
\sum_i \left(\sum_t \frac{s_{i,t}+1}{2}-1\right)^2
+
\sum_t \left(\sum_i \frac{s_{i,t}+1}{2}-1\right)^2
$$

Since in this problem the binary formulation is more natural, we can implement it using ``dimod.BINARY`` and if we want to change to the spin formulation, we can easily do it using ``.change_vartype()``.

In [15]:
# Define the cities and distances between them
cities = ["A", "B", "C", "D", "E", "F"]

n = len(cities)
time = range(n)

dist = {
    "A": {"B": 4, "C": 2, "D": 7, "E": 3, "F": 9},
    "B": {"A": 4, "C": 5, "D": 3, "E": 6, "F": 8},
    "C": {"A": 2, "B": 5, "D": 4, "E": 3, "F": 7},
    "D": {"A": 7, "B": 3, "C": 4, "E": 2, "F": 5},
    "E": {"A": 3, "B": 6, "C": 3, "D": 2, "F": 4},
    "F": {"A": 9, "B": 8, "C": 7, "D": 5, "E": 4}
}

# Create a empty binary quadratic model for the TSP using binary variables
bqm = dimod.BinaryQuadraticModel({}, {}, 0.0, dimod.BINARY)

# Add the distance terms to the BQM
for u in cities:
    for v in cities:
        if u != v:
            for t in time:
                next_t = (t + 1) % n
                origin = f"x_{u}_{t}"
                dest = f"x_{v}_{next_t}"
                bqm.add_quadratic(origin, dest, dist[u][v])


# Add retrictions
penalization = 10

# Each time step must have exactly one city assigned
for u in cities:
    terms = []
    for t in time:
        var = f"x_{u}_{t}"
        terms.append((var, 1))

    bqm.add_linear_equality_constraint(
        terms,
        lagrange_multiplier=penalization,
        constant=-1
    )

# Each city must be visited exactly once
for t in time:
    terms = []
    for u in cities:
        var = f"x_{u}_{t}"
        terms.append((var, 1))

    bqm.add_linear_equality_constraint(
        terms,
        lagrange_multiplier=penalization,
        constant=-1
    )

In [16]:
# Use the Simulated Annealing Sampler to find a solution
sampler = SimulatedAnnealingSampler()
solution = sampler.sample(bqm, num_reads=1000)

# Aggregate the results to find the best solution
solution = solution.aggregate()
best = solution.first
best_energy = best.energy
df = solution.to_pandas_dataframe()
df_sorted = df.sort_values("energy", ascending=True)

# Print the best energy and the first 5 solutions
print(30 * "#")
print("Best energy found:", best_energy)
print(30 * "#")
print("First 5 solutions:")
print(df_sorted.head(5))

##############################
Best energy found: 21.0
##############################
First 5 solutions:
     x_A_0  x_A_1  x_A_2  x_A_3  x_A_4  x_A_5  x_B_0  x_B_1  x_B_2  x_B_3  \
17       0      0      0      0      1      0      0      0      0      1   
192      0      0      1      0      0      0      0      1      0      0   
113      0      1      0      0      0      0      0      0      1      0   
42       0      0      0      0      0      1      0      0      0      0   
60       1      0      0      0      0      0      0      1      0      0   

     ...  x_E_4  x_E_5  x_F_0  x_F_1  x_F_2  x_F_3  x_F_4  x_F_5  energy  \
17   ...      0      0      0      1      0      0      0      0    21.0   
192  ...      1      0      0      0      0      0      0      1    21.0   
113  ...      0      1      0      0      0      0      1      0    21.0   
42   ...      0      0      0      0      1      0      0      0    21.0   
60   ...      1      0      0      0      0      1  

The simulated annealing sampler found several configurations with the same minimum energy, $E=21$. Unlike Max-Cut, the solution is not easy to interpret directly because the TSP formulation uses many more variables. For this reason, we define a function to reconstruct and print the route, making it easier to analyze the result.

In [17]:
# Translate the solution into a readable route
def print_route(df_sorted, cities, time, idx) -> None:
    route_steps = []

    # .iloc[idx] selects the position
    current_sol = df_sorted.iloc[idx]

    for city in cities:
        for t in time:
            var = f"x_{city}_{t}"
            if var in current_sol and current_sol[var] == 1:
                route_steps.append((int(t), city))

    # Sort cronologically
    route_steps.sort(key=lambda x: x[0])

    # Extract cities names
    route = [step[1] for step in route_steps]

    # Show the route
    print(f"Route (idx #{idx}):")
    print("Energy:", current_sol["energy"])
    print(" -> ".join(route))

In [18]:
# Print best 5 routes
print(30 * "#")
for i in range(5):
  print_route(df_sorted, cities, time, i)
  print(30 * "#")


##############################
Route (idx #0):
Energy: 21.0
E -> F -> D -> B -> A -> C
##############################
Route (idx #1):
Energy: 21.0
D -> B -> A -> C -> E -> F
##############################
Route (idx #2):
Energy: 21.0
C -> A -> B -> D -> F -> E
##############################
Route (idx #3):
Energy: 21.0
C -> E -> F -> D -> B -> A
##############################
Route (idx #4):
Energy: 21.0
A -> B -> D -> F -> E -> C
##############################


If we look closely, we can see that all these solutions correspond to the same route, but with a different starting city or travelling direction. This happens because, in the TSP, the same cycle can be represented in several equivalent ways.

If we use the spin formulation instead of the binary formulation, we can verify that the solution remains the same. This is expected because both formulations are equivalent and can be transformed into each other through the change of variables between binary and spin variables.

In [20]:
bqm_spin = bqm.change_vartype(dimod.SPIN, inplace=False)

solution_spin = sampler.sample(bqm_spin, num_reads=1000)
solution_spin = solution_spin.aggregate()
best_spin = solution_spin.first
best_energy_spin = best_spin.energy
df_spin = solution_spin.to_pandas_dataframe()
df_spin_sorted = df_spin.sort_values("energy", ascending=True)

print(30 * "#")
print("Best energy found:", best_energy_spin)
print(30 * "#")
print("First 5 solutions:")
print(df_spin_sorted.head(5))

##############################
Best energy found: 21.0
##############################
First 5 solutions:
    x_A_0  x_A_1  x_A_2  x_A_3  x_A_4  x_A_5  x_B_0  x_B_1  x_B_2  x_B_3  ...  \
47     -1     -1      1     -1     -1     -1     -1      1     -1     -1  ...   
37     -1     -1     -1     -1      1     -1     -1     -1     -1      1  ...   
66     -1     -1     -1     -1     -1      1     -1     -1     -1     -1  ...   
50     -1     -1     -1      1     -1     -1     -1     -1     -1     -1  ...   
75     -1      1     -1     -1     -1     -1     -1     -1      1     -1  ...   

    x_E_4  x_E_5  x_F_0  x_F_1  x_F_2  x_F_3  x_F_4  x_F_5  energy  \
47      1     -1     -1     -1     -1     -1     -1      1    21.0   
37     -1     -1     -1      1     -1     -1     -1     -1    21.0   
66     -1     -1     -1     -1      1     -1     -1     -1    21.0   
50     -1     -1      1     -1     -1     -1     -1     -1    21.0   
75     -1      1     -1     -1     -1     -1      1     -1

In [21]:
print(30 * "#")
for i in range(5):
  print_route(df_spin_sorted, cities, time, i)
  print(30 * "#")


##############################
Route (idx #0):
Energy: 21.0
D -> B -> A -> C -> E -> F
##############################
Route (idx #1):
Energy: 21.0
E -> F -> D -> B -> A -> C
##############################
Route (idx #2):
Energy: 21.0
C -> E -> F -> D -> B -> A
##############################
Route (idx #3):
Energy: 21.0
F -> E -> C -> A -> B -> D
##############################
Route (idx #4):
Energy: 21.0
C -> A -> B -> D -> F -> E
##############################


As expected, we get the exact same routes as before.